In [ ]:
# Set the environment variable so that only GPU 5 is visible
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3"

In [ ]:
import json
import pandas as pd
import numpy as np
import requests
import time
from huggingface_hub import login
from sklearn.decomposition import PCA
from matplotlib import pyplot as plt

In [ ]:
from numpy import dot
from numpy.linalg import norm

# Calculate angle between two vectors layerwise
def angle_between_vectors(R1, R2):
    angle_across_layers = []
    for layer in range(R1.shape[0]):  # Iterate over layers
        r1_layer = R1[layer]
        r2_layer = R2[layer]
        magnitude_r1 = np.linalg.norm(r1_layer)
        magnitude_r2 = np.linalg.norm(r2_layer)
        cos_theta = dot(r1_layer, r2_layer) / (magnitude_r1 * magnitude_r2)
        cos_theta = np.clip(cos_theta, -1.0, 1.0)
        angle_radians = np.arccos(cos_theta)
        angle_degrees = np.degrees(angle_radians)
        angle_across_layers.append(angle_degrees)
    return angle_across_layers  

# Calculate cosine similarity layer by layer
def get_similarity_and_magnitude(R1, R2):
    cosine_similarities = []
    magnitude_difference = []
    mag_r1, mag_r2 = [], []
    for layer in range(R1.shape[0]):  # Iterate over layers
        r1_layer = R1[layer]
        r2_layer = R2[layer]
        cosine_similarity = dot(r1_layer, r2_layer) / (norm(r1_layer) * norm(r2_layer))
        cosine_similarities.append(cosine_similarity)

        magnitude_r1 = np.linalg.norm(r1_layer)
        magnitude_r2 = np.linalg.norm(r2_layer)
        mag_r1.append(magnitude_r1)
        mag_r2.append(magnitude_r2)

        diff = magnitude_r2 - magnitude_r1
        if diff < 0:
            diff = -diff
        # print(diff)
        magnitude_difference.append(diff)

    return cosine_similarities, magnitude_difference, mag_r1, mag_r2

In [ ]:
pref = "/home/gshah010/project_cross_role/FRESH START/Visualization/Vectors/QWEN/AdvBench"
A = np.load(f"{pref}/vector_A.npy")
img_pos_B = np.load(f"{pref}/img in position/vector_B.npy")
img_pos_C = np.load(f"{pref}/img in position/vector_C.npy")
B = np.load(f"{pref}/img outside/vector_B.npy")
C = np.load(f"{pref}/img outside/vector_C.npy")
newB = np.load(f"{pref}/img at end/vector_B.npy")
newC = np.load(f"{pref}/img at end/vector_C.npy")
refusal = np.load(f"{pref}/refusal_vector.npy")

In [ ]:
A.shape, B.shape, newB.shape, newC.shape, refusal.shape, C.shape,

In [ ]:
random_array = np.random.rand(28, 3584)  # Values between 0 and 1
print(random_array.shape)  # Output: (28, 3584)

In [ ]:
new_attack_vectors = [A, img_pos_B, img_pos_C, B, C, newB, newC, random_array, -refusal] 
# new_attack_success_vectors = [AIM_sucess_vec, prefix_success_vec, refusal_success_vec, style_success_vec, evil_success_vec, payload_success_vec, GCG_success_vec] 
new_attack_names = ["swap", "img pos", "img pos_swap", "img out", "img out_swap", "img end", "img end_swap", "random", "refusal"] 

# role_attack_names = ["No Image Cross Role (A)", "Img Out No Swap (B)", "Img Out Cross Role (C)", "Img End (B)"]
# role_attack_success_vectors = [no_img_cross_role_vec, img_out_no_swap_vec, img_out_cross_role_vec, ]
# role_attack_vectors = [A, B, C, img_end_B]

# similarity calculation
def get_similarities(new_attack_vectors, target_vector):
    new_attack_similarity_list, new_attack_magnitude_list, new_attack_angle_list = [], [], []
    for vecA in new_attack_vectors:
        A_to_B_sim, A_to_B_mag, A_mag, B_mag = get_similarity_and_magnitude(vecA, target_vector)
        angle_A_to_B = angle_between_vectors(vecA, target_vector)
        new_attack_angle_list.append(angle_A_to_B)
        new_attack_similarity_list.append(A_to_B_sim)
        new_attack_magnitude_list.append(A_to_B_mag)

    # role_attack_similarity_list, role_attack_magnitude_list, role_attack_angle_list = [], [], []
    # for vecA in role_attack_vectors:
    #     A_to_B_sim, A_to_B_mag, A_mag, B_mag = get_similarity_and_magnitude(vecA, target_vector)
    #     angle_A_to_B = angle_between_vectors(vecA, target_vector)
    #     role_attack_angle_list.append(angle_A_to_B)
    #     role_attack_similarity_list.append(A_to_B_sim)
    #     role_attack_magnitude_list.append(A_to_B_mag)

    return new_attack_similarity_list, new_attack_magnitude_list, new_attack_angle_list #, role_attack_similarity_list, role_attack_magnitude_list, role_attack_angle_list

In [ ]:
import itertools
import pickle
markers = itertools.cycle(['o', 'x', '*', '.', '1', 'p', 'X'])
colors = itertools.cycle(plt.cm.tab10.colors)  # Using tab10 colormap for distinct colors
colors2=['red', 'darkviolet', 'lime', 'darkgreen', 'silver', 'orange', 'blue', 'maroon', 'black']

# Plot cosine similarities
def plot_all_vectors_in_similarity_figure(save, figure_save_path, new_attack_sim_list, new_attack_names, message):
    layers = range(1, len(new_attack_sim_list[0]) + 1)
    plt.figure(figsize=(10, 8))
    
    for i, similarity in enumerate(new_attack_sim_list):
        plt.plot(layers, similarity, marker='.', color=colors2[i], label=new_attack_names[i])
    
    # for i, similarity in enumerate(role_attack_sim_list):
    #     plt.plot(layers, similarity, marker='.', color=colors2[i], label=role_attack_names[i])
    
    plt.xlabel('Layer')
    plt.ylabel('Cosine Similarity')
    plt.title(message)
    plt.axhline(0, color='gray', linestyle='--', linewidth=0.8)  # Reference line at y=0
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.legend()
    
    if save: 
        plt.savefig(figure_save_path, dpi=100)
    plt.show()


# Final plot setting
def final_plot_all_vectors_in_similarity_figure(save, figure_save_path, new_attack_sim_list, new_attack_names, message):
    layers = range(1, len(new_attack_sim_list[0]) + 1)
    plt.figure(figsize=(10, 8))

    # Increase the size of x-ticks and y-ticks
    plt.tick_params(axis='x', labelsize=20)  # Set x-tick label size
    plt.tick_params(axis='y', labelsize=20)  # Set y-tick label size
    
    for i, similarity in enumerate(new_attack_sim_list):
        plt.plot(layers, similarity, linestyle='-', marker=None, color=colors2[i], label=new_attack_names[i])
    
    # for i, similarity in enumerate(role_attack_sim_list):
    #     plt.plot(layers, similarity, marker='.', color=colors2[i], label=role_attack_names[i])
    
    plt.xlabel('Layer', fontsize=20)
    plt.ylabel('Cosine Similarity', fontsize=20)
    # plt.title(message, fontsize=16, pad=80)
    # plt.axhline(0, color='gray', linestyle='--', linewidth=0.8)  # Reference line at y=0
    plt.grid(False)
    # plt.legend(loc='upper center', bbox_to_anchor=(0.5, 1.155), ncol=4, markerscale=5, fontsize=17)

    # Ensure x-ticks start from layer 1
    plt.xticks(range(1, len(new_attack_sim_list[0]) + 1, 5))  
    
    if save: 
        plt.savefig(figure_save_path, dpi=100)
    plt.show()


# Plot angles
def plot_all_magnitudes_in_same_figure(save, figure_save_path, new_attack_magnitude, new_attack_names, role_attack_magnitude, role_attack_names, message):
    layers = range(1, len(new_attack_magnitude[0]) + 1)
    plt.figure(figsize=(10, 8))
    
    for i, magnitude in enumerate(new_attack_magnitude):
        plt.plot(layers, magnitude, marker='.', color=next(colors), label=new_attack_names[i])
    
    for i, magnitude in enumerate(role_attack_magnitude):
        plt.plot(layers, magnitude, marker='.', color=colors2[i], label=role_attack_names[i])
    
    plt.xlabel('Layer')
    plt.ylabel('Magnitude')
    plt.title(message)
    plt.axhline(0, color='gray', linestyle='--', linewidth=0.8)  # Reference line at y=0
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.legend()
    
    if save: 
        plt.savefig(figure_save_path, dpi=100)
    plt.show()


# Plot angles
def plot_all_angles_in_same_figure(save, figure_save_path, new_attack_angles, new_attack_names, role_attack_angles, role_attack_names, message):
    layers = range(1, len(new_attack_angles[0]) + 1)
    plt.figure(figsize=(10, 8))
    
    for i, angle in enumerate(new_attack_angles):
        plt.plot(layers, angle, marker='.', color=next(colors), label=new_attack_names[i])
    
    for i, angle in enumerate(role_attack_angles):
        plt.plot(layers, angle, marker='.', color=colors2[i], label=role_attack_names[i])
    
    plt.xlabel('Layer')
    plt.ylabel('Degrees')
    plt.title(message)
    plt.axhline(0, color='gray', linestyle='--', linewidth=0.8)  # Reference line at y=0
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.legend()
    
    if save: 
        plt.savefig(figure_save_path, dpi=100)
    plt.show()

In [ ]:
# figure_folder_path = "/home/gshah010/project_cross_role/FRESH START/Other Attacks/plot/"
figure_folder_path = pref+"/"
fig_save = False

new_sim, new_mag, new_ang = get_similarities(new_attack_vectors, -refusal)
final_plot_all_vectors_in_similarity_figure(fig_save, figure_folder_path+"QWEN_cosine_rf_fourth.pdf", new_sim, new_attack_names,  "QWEN")
# plot_all_magnitudes_in_same_figure(True, figure_folder_path+"attack_mag_vs_refusal.png", new_mag, new_attack_names, role_mag, role_attack_names, "Magnitude Comparison between Refusal and Attack Vectors")
# plot_all_angles_in_same_figure(True, figure_folder_path+"attack_angle_vs_refusal.png", new_ang, new_attack_names, role_ang, role_attack_names, "Layer wise angle calculation on Refusal Direction")

# new_sim, new_mag, new_ang = get_similarities(new_attack_vectors, A)
# plot_all_vectors_in_similarity_figure(fig_save, figure_folder_path+"attack_sim_vs_vector_A.png", new_sim, new_attack_names,  "Cosine Similarity between A and Attack Vectors")
# plot_all_magnitudes_in_same_figure(True, figure_folder_path+"attack_mag_vs_vector_A.png", new_mag, new_attack_names, role_mag, role_attack_names, "Magnitude Comparison between A and Attack Vectors")
# plot_all_angles_in_same_figure(True, figure_folder_path+"attack_angle_vs_vector_A.png", new_ang, new_attack_names, role_ang, role_attack_names, "Layer wise angle calculation on A")

# new_sim, new_mag, new_ang = get_similarities(new_attack_vectors, B)
# plot_all_vectors_in_similarity_figure(fig_save, figure_folder_path+"attack_sim_vs_vector_B.png", new_sim, new_attack_names,  "Cosine Similarity between B and Attack Vectors")
# plot_all_magnitudes_in_same_figure(True, figure_folder_path+"attack_mag_vs_vector_B.png", new_mag, new_attack_names, role_mag, role_attack_names, "Magnitude Comparison between B and Attack Vectors")
# plot_all_angles_in_same_figure(True, figure_folder_path+"attack_angle_vs_vector_B.png", new_ang, new_attack_names, role_ang, role_attack_names, "Layer wise angle calculation on B")

# new_sim, new_mag, new_ang = get_similarities(new_attack_vectors, C)
# plot_all_vectors_in_similarity_figure(fig_save, figure_folder_path+"attack_sim_vs_vector_C.png", new_sim, new_attack_names, "Cosine Similarity between C and Attack Vectors")
# plot_all_magnitudes_in_same_figure(True, figure_folder_path+"attack_mag_vs_vector_C.png", new_mag, new_attack_names, role_mag, role_attack_names, "Magnitude Comparison between C and Attack Vectors")
# plot_all_angles_in_same_figure(True, figure_folder_path+"attack_angle_vs_vector_C.png", new_ang, new_attack_names, role_ang, role_attack_names, "Layer wise angle calculation on C")

# new_sim, new_mag, new_ang = get_similarities(new_attack_vectors, newB)
# plot_all_vectors_in_similarity_figure(fig_save, figure_folder_path+"attack_sim_vs_vector_newB.png", new_sim, new_attack_names, "Cosine Similarity between newB and Attack Vectors")

# new_sim, new_mag, new_ang = get_similarities(new_attack_vectors, newC)
# plot_all_vectors_in_similarity_figure(fig_save, figure_folder_path+"attack_sim_vs_vector_newC.png", new_sim, new_attack_names,  "Cosine Similarity between newC and Attack Vectors")

In [ ]:
for x in new_attack_vectors:
    new_sim, new_mag, new_ang = get_similarities(new_attack_vectors, x)

    for i in range(len(new_sim)):
        if i == len(new_sim)-1:
            continue 

        sim_list = new_sim[i]
        # print(sim_list)
        print(np.mean(sim_list))
    print("#"*50)

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
from pdf2image import convert_from_path
import pickle

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from pdf2image import convert_from_path
import pickle

# Load the saved PDF images
img1 = np.array(convert_from_path("/home/gshah010/project_cross_role/FRESH START/Visualization/Vectors/QWEN/AdvBench/QWEN_cosine_rf.pdf")[0])
img2 = np.array(convert_from_path("/home/gshah010/project_cross_role/FRESH START/Visualization/Vectors/LLAVA/AdvBench/LLAVA_cosine_rf.pdf")[0])
img3 = np.array(convert_from_path("/home/gshah010/project_cross_role/FRESH START/Visualization/Vectors/PHI/AdvBench/PHI_cosine_rf.pdf")[0])

# Load the legend data from the pickle file
with open("/home/gshah010/project_cross_role/FRESH START/Visualization/Vectors/QWEN/legend_data.pkl", "rb") as f:
    handles, labels = pickle.load(f)

# Create figure and axes for 1 row, 3 columns
fig, axes = plt.subplots(1, 3, figsize=(25, 5))

# Display each image in a subplot
axes[0].imshow(img1)
axes[1].imshow(img2)
axes[2].imshow(img3)

# Set the titles below each image
axes[0].set_xlabel("(a) Qwen2-VL-7B-Instruct", fontsize=19, labelpad=20)
axes[1].set_xlabel("(b) llava-1.5-7b-hf", fontsize=19, labelpad=20)
axes[2].set_xlabel("(c) Phi-3.5-vision instruct", fontsize=19, labelpad=20)

# Remove the axis labels and ticks
for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_frame_on(False)  # Removes only the box but keeps labels

# Create a common legend for the figure
fig.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, 1.05), ncol=5, fontsize=17)

# Adjust spacing
fig.subplots_adjust(wspace=-0.5, hspace=-0.5)

# Save the combined figure as a PDF (do this before calling plt.show())
save_path = "/home/gshah010/project_cross_role/FRESH START/Visualization/Vectors/merged_image.pdf"
plt.savefig(save_path, dpi=100, bbox_inches="tight")

# Close the figure to avoid it showing again when plt.show() is called
plt.close(fig)

# Now show the combined figure after saving
# Re-create the figure with the three images
fig, axes = plt.subplots(1, 3, figsize=(25, 5))

# Display each image in a subplot again (without legend)
axes[0].imshow(img1)
axes[1].imshow(img2)
axes[2].imshow(img3)

# Set the titles below each image
axes[0].set_xlabel("(a) Qwen2-VL-7B-Instruct", fontsize=19, labelpad=20)
axes[1].set_xlabel("(b) llava-1.5-7b-hf", fontsize=19, labelpad=20)
axes[2].set_xlabel("(c) Phi-3.5-vision instruct", fontsize=19, labelpad=20)

# Remove the axis labels and ticks
for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_frame_on(False)

# Create the common legend again (this time without affecting previous figure)
fig.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, 1.05), ncol=5, fontsize=17)

# Adjust spacing
fig.subplots_adjust(wspace=-0.5, hspace=-0.5)

# Now show the final figure
plt.show()


In [ ]:
%pip install fpdf2

In [ ]:
from fpdf import FPDF
pdf = FPDF()
# imagelist is the list with all image filenames

for image in imagelist:
    pdf.add_page()
    pdf.image(image,x,y,w,h)
pdf.output("yourfile.pdf", "F")